# 03 — Modeling: Pipelines, Baselines, Cross-Validation, Tuning, Imbalance

Goal: build a leakage-safe pipeline, establish baselines, compare model
families under cross-validation, tune the strongest finalists, and compare
imbalance-handling strategies — all without ever touching the test set.

This notebook mirrors `src/train.py`, but broken into inspectable steps.
For the fully automated version, run: `python -m src.train --synthetic`


In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate

from src.data import TARGET_COLUMN, get_feature_columns, RANDOM_SEED
from src.features import build_full_pipeline, build_imbalanced_pipeline
from src.evaluate import evaluate_at_threshold

np.random.seed(RANDOM_SEED)

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")

numerical_columns, categorical_columns = get_feature_columns(train_df)
feature_cols = numerical_columns + categorical_columns

X_train, y_train = train_df[feature_cols], train_df[TARGET_COLUMN]
X_val, y_val = val_df[feature_cols], val_df[TARGET_COLUMN]

print(f"Numerical: {numerical_columns}")
print(f"Categorical: {categorical_columns}")


Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 1. Preprocessing pipeline

`ColumnTransformer` with two branches:
- **Numerical:** `SimpleImputer(median)` → `StandardScaler` (harmless for
  trees, required for logistic regression's coefficient magnitudes to be
  comparable and for regularization to behave sensibly).
- **Categorical:** `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown="ignore")`.
  `handle_unknown="ignore"` means a category never seen during training
  (e.g. a brand-new `PaymentMethod` added by the business later) is encoded
  as all-zeros at inference instead of raising an error — the service stays
  available rather than crashing on novel input.

Because this preprocessing is wrapped inside `sklearn.pipeline.Pipeline`, all
of its learned statistics (medians, most-frequent categories, one-hot
vocabulary, scaler mean/std) are fit **only** on whatever data `.fit()` is
called with — i.e. only training folds during cross-validation. This is
proven directly in `tests/test_pipeline.py::test_preprocessing_only_learns_from_training_fold`.


In [2]:
demo_pipeline = build_full_pipeline(LogisticRegression(max_iter=1000), numerical_columns, categorical_columns)
demo_pipeline


c:\Users\arad_\OneDrive\Desktop\ML\Customer-Churn-Prediction\telco-churn-project\.venv\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:203: UserWarning: potentially wrong underline length... 
Edge cases handled 
------------------- in 
Adds a handful of domain-specific engineered features on top of the raw
columns: how much a customer pays relative to how long they have stayed,...
  while not self._is_at_section() and not self._doc.eof():
c:\Users\arad_\OneDrive\Desktop\ML\Customer-Churn-Prediction\telco-churn-project\.venv\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section New Features
  self[section] = content
c:\Users\arad_\OneDrive\Desktop\ML\Customer-Churn-Prediction\telco-churn-project\.venv\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Edge Cases Handled
  self[section] = content


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('domain_features', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,service_columns,"['OnlineSecurity', 'OnlineBackup', ...]"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical or

## 2. Baselines: DummyClassifier and Logistic Regression

In [3]:
dummy = build_full_pipeline(DummyClassifier(strategy="stratified", random_state=RANDOM_SEED),
                             numerical_columns, categorical_columns)
dummy.fit(X_train, y_train)
dummy_proba = dummy.predict_proba(X_val)[:, 1]
dummy_metrics = evaluate_at_threshold(y_val, dummy_proba)
print("Dummy baseline:", dummy_metrics)


Dummy baseline: ClassificationMetrics(threshold=0.5, accuracy=0.6121097445600757, precision=0.26881720430107525, recall=0.2669039145907473, f1=0.26785714285714285, roc_auc=0.5020086583263015, pr_auc=0.26663956565101576, brier=0.38789025543992434, tn=572, fp=204, fn=206, tp=75, prediction_latency_ms=0.0)


In [4]:
logreg = build_full_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
                              numerical_columns, categorical_columns)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_val)[:, 1]
logreg_metrics = evaluate_at_threshold(y_val, logreg_proba)
print("Logistic Regression:", logreg_metrics)


Logistic Regression: ClassificationMetrics(threshold=0.5, accuracy=0.8032166508987701, precision=0.6697674418604651, recall=0.5124555160142349, f1=0.5806451612903226, roc_auc=0.8360283596874197, pr_auc=0.640895224085633, brier=0.13822063546294056, tn=705, fp=71, fn=137, tp=144, prediction_latency_ms=0.0)


**Does the model outperform a trivial strategy?** Compare `logreg_metrics.roc_auc`
to `dummy_metrics.roc_auc` — the dummy classifier is expected to sit near 0.5
(random ranking), so any meaningful lift above that confirms the features
carry real signal.


In [5]:
coefs = pd.DataFrame({
    "feature": logreg.named_steps["preprocessor"].get_feature_names_out(),
    "coefficient": logreg.named_steps["classifier"].coef_.ravel(),
}).sort_values("coefficient", key=abs, ascending=False)
coefs.head(15)


,feature,coefficient
42,cat__Contract_Two year,-0.918595
40,cat__Contract_Month-to-month,0.725927
19,cat__InternetService_DSL,-0.650537
1,num__tenure,-0.565031
20,cat__InternetService_Fiber optic,0.559734
2,num__MonthlyCharges,-0.365482
43,cat__PaperlessBilling_No,-0.334337
6,num__has_internet,0.298377
16,cat__MultipleLines_No,-0.277132
51,cat__tenure_bucket_25-48mo,-0.274431


**Are coefficient magnitudes directly comparable after one-hot encoding and
scaling?** Only partially: numeric features were standardized so their
coefficients are on a comparable "per standard deviation" scale, but one-hot
encoded coefficients represent a shift relative to their (implicit) baseline
category, which is *not* on the same scale as a per-SD numeric effect. Use
these for relative ranking within a feature group, not for direct
cross-type magnitude comparison.


## 3. Nonlinear models: decision tree, random forest, gradient boosting

In [6]:
candidates = {
    "decision_tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_SEED),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1),
    "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_SEED),
}

comparison_rows = [
    {"model": "dummy", "val_roc_auc": dummy_metrics.roc_auc, "val_pr_auc": dummy_metrics.pr_auc},
    {"model": "logistic_regression", "val_roc_auc": logreg_metrics.roc_auc, "val_pr_auc": logreg_metrics.pr_auc},
]
fitted_models = {"logistic_regression": logreg}

for name, est in candidates.items():
    pipe = build_full_pipeline(est, numerical_columns, categorical_columns)
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_val)[:, 1]
    m = evaluate_at_threshold(y_val, proba)
    comparison_rows.append({"model": name, "val_roc_auc": m.roc_auc, "val_pr_auc": m.pr_auc})
    fitted_models[name] = pipe

comparison_table = pd.DataFrame(comparison_rows).sort_values("val_roc_auc", ascending=False).reset_index(drop=True)
comparison_table


,model,val_roc_auc,val_pr_auc
0,gradient_boosting,0.836884,0.643390
1,logistic_regression,0.836028,0.640895
2,random_forest,0.831011,0.635426
3,decision_tree,0.813589,0.577432
4,dummy,0.502009,0.266640


**Which model is strongest before tuning?** Read off the top row of
`comparison_table` by `val_roc_auc`. **Why might impurity-based importance
favor high-cardinality features?** Impurity reduction is measured on splits;
features with many possible split points (numeric, or one-hot-exploded
high-cardinality categoricals) get more chances to appear "important" purely
from having more candidate thresholds, independent of true predictive value —
this is why we cross-check with permutation importance in notebook 04.


## 4. Cross-validation and tuning of the top two finalists

In [7]:
finalists = [m for m in comparison_table["model"] if m != "dummy"][:2]
print("Finalists:", finalists)

param_grids = {
    "logistic_regression": {"classifier__C": [0.01, 0.1, 0.3, 1.0, 3.0, 10.0]},
    "random_forest": {
        "classifier__n_estimators": [200, 300, 400, 600],
        "classifier__max_depth": [4, 6, 8, 10, None],
        "classifier__min_samples_leaf": [1, 2, 4, 8],
    },
    "gradient_boosting": {
        "classifier__n_estimators": [100, 150, 200, 300],
        "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "classifier__max_depth": [2, 3, 4],
    },
    "decision_tree": {
        "classifier__max_depth": [3, 4, 5, 6, 8, 10],
        "classifier__min_samples_leaf": [1, 2, 4, 8, 16],
    },
}

all_candidates = {"logistic_regression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED), **candidates}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

tuned = {}
for name in finalists:
    pipe = build_full_pipeline(all_candidates[name], numerical_columns, categorical_columns)
    search = RandomizedSearchCV(
        pipe, param_distributions=param_grids.get(name, {}), n_iter=15,
        scoring="roc_auc", cv=cv, random_state=RANDOM_SEED, n_jobs=-1, refit=True,
    )
    search.fit(X_train, y_train)
    tuned[name] = search
    std = search.cv_results_["std_test_score"][search.best_index_]
    print(f"{name}: best CV ROC-AUC = {search.best_score_:.4f} +/- {std:.4f} | params = {search.best_params_}")


Finalists: ['gradient_boosting', 'logistic_regression']
gradient_boosting: best CV ROC-AUC = 0.8477 +/- 0.0037 | params = {'classifier__n_estimators': 200, 'classifier__max_depth': 2, 'classifier__learning_rate': 0.05}
logistic_regression: best CV ROC-AUC = 0.8489 +/- 0.0050 | params = {'classifier__C': 0.1}


c:\Users\arad_\OneDrive\Desktop\ML\Customer-Churn-Prediction\telco-churn-project\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 6 is smaller than n_iter=15. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


**Did tuning produce a meaningful improvement or only noise?** Compare each
finalist's tuned CV ROC-AUC (and its +/- standard deviation across folds) to
its untuned validation score in `comparison_table`. If the improvement is
smaller than roughly one standard deviation of the CV scores, treat it as
noise rather than a real gain. **How would nested CV change confidence?**
Nested CV would use an inner loop for hyperparameter selection and a
completely separate outer loop for scoring, removing the mild optimism bias
that comes from selecting hyperparameters using the same folds you're
scoring. This project does **not** use nested CV: the tuned CV scores above
come from the same folds that picked the hyperparameters, so they are
slightly optimistic. The untouched test set is the honest estimate.


## 5. Imbalance strategies

In [8]:
from src.features import build_imbalanced_pipeline

best_finalist_name = finalists[0]
base_params = tuned[best_finalist_name].best_params_
base_est = all_candidates[best_finalist_name]

imbalance_rows = []

# (a) No treatment
pipe_none = build_imbalanced_pipeline(base_est, numerical_columns, categorical_columns, resampler="none")
pipe_none.fit(X_train, y_train)
m = evaluate_at_threshold(y_val, pipe_none.predict_proba(X_val)[:, 1])
imbalance_rows.append({"strategy": "none", "roc_auc": m.roc_auc, "recall@0.5": m.recall, "precision@0.5": m.precision})

# (b) class_weight="balanced" (only for estimators that support it)
if hasattr(base_est, "class_weight"):
    est_bal = all_candidates[best_finalist_name].__class__(**{**base_est.get_params(), "class_weight": "balanced"})
    pipe_bal = build_imbalanced_pipeline(est_bal, numerical_columns, categorical_columns, resampler="class_weight")
    pipe_bal.fit(X_train, y_train)
    m = evaluate_at_threshold(y_val, pipe_bal.predict_proba(X_val)[:, 1])
    imbalance_rows.append({"strategy": "class_weight_balanced", "roc_auc": m.roc_auc, "recall@0.5": m.recall, "precision@0.5": m.precision})
else:
    print(f"{best_finalist_name} does not support class_weight; skipping that comparison arm.")

# (c) SMOTE / RandomOverSampler — only if imbalanced-learn is installed.
# IMPORTANT: resampling must occur INSIDE the pipeline so cross-validation
# and the held-out fold never see synthetic samples derived from validation
# data — imblearn.pipeline.Pipeline enforces this automatically.
try:
    import imblearn  # noqa: F401
    for resampler in ["smote", "random_oversample"]:
        pipe_r = build_imbalanced_pipeline(base_est, numerical_columns, categorical_columns, resampler=resampler)
        pipe_r.fit(X_train, y_train)
        m = evaluate_at_threshold(y_val, pipe_r.predict_proba(X_val)[:, 1])
        imbalance_rows.append({"strategy": resampler, "roc_auc": m.roc_auc, "recall@0.5": m.recall, "precision@0.5": m.precision})
except ImportError:
    print("imbalanced-learn not installed in this environment — install with "
          "`pip install imbalanced-learn` to run the SMOTE / RandomOverSampler arms. "
          "(Code path is fully implemented in src/features.py::build_imbalanced_pipeline.)")

pd.DataFrame(imbalance_rows)


gradient_boosting does not support class_weight; skipping that comparison arm.


,strategy,roc_auc,recall@0.5,precision@0.5
0,none,0.836884,0.505338,0.682692
1,smote,0.836102,0.647687,0.608696
2,random_oversample,0.837425,0.779359,0.530266


**Why must synthetic samples never be created before the split?** SMOTE
interpolates new synthetic points between real minority-class neighbors. If
applied before splitting (or across CV folds), synthetic points derived from
what becomes "validation" data could be interpolated using — or interpolated
*into* — the training set, letting validation information leak into what the
model learns, producing optimistic and invalid evaluation. Placing the
resampler as a pipeline step guarantees `imblearn.pipeline.Pipeline` only
resamples the active training fold on each `.fit()` call.

**Which strategy maximizes recall vs. lowest business cost?** Compare the
table above; typically `class_weight_balanced` and `SMOTE` raise recall by
trading away some precision. Which one is actually *cheapest* depends on the
false-negative/false-positive cost ratio — that's decided in notebook 04,
not here, to avoid choosing a strategy based on accuracy-flavored intuition.


## 6. Select final model for downstream threshold optimization

In [9]:
best_name = max(tuned, key=lambda k: tuned[k].best_score_)
final_pipeline = tuned[best_name].best_estimator_
print(f"Selected model: {best_name} (CV ROC-AUC = {tuned[best_name].best_score_:.4f})")

final_proba_val = final_pipeline.predict_proba(X_val)[:, 1]
final_metrics = evaluate_at_threshold(y_val, final_proba_val)
print(final_metrics)


Selected model: logistic_regression (CV ROC-AUC = 0.8489)
ClassificationMetrics(threshold=0.5, accuracy=0.804162724692526, precision=0.6761904761904762, recall=0.505338078291815, f1=0.5784114052953157, roc_auc=0.8361338371794402, pr_auc=0.6405296733951744, brier=0.13816536965433593, tn=708, fp=68, fn=139, tp=142, prediction_latency_ms=0.0)


## Exit criterion
✅ One command (`build_full_pipeline(...).fit(...)`) fits and evaluates a
leakage-safe baseline; multiple model families compared under CV; finalists
tuned with documented CV evidence; imbalance strategies compared without
contaminating folds. The chosen `final_pipeline` is carried forward to
`04_decision_analysis.ipynb` for threshold optimization and interpretation.
